# Overview

This notebook demonstrates how to scan for TF binding motifs. The base GRN will be generated by combining the ATAC-seq peaks and motif information.

### Notebook file
Notebook file is available on CellOracle's GitHub page.
https://github.com/morris-lab/CellOracle/blob/master/docs/notebooks/02_motif_scan/02_atac_peaks_to_TFinfo_with_celloracle_20200801.ipynb


# 0. Import libraries

In [1]:
# 必须在运行 scan 之前运行
from gimmemotifs.config import MotifConfig
# 强制设置为单核，避免集群死锁、
MotifConfig().set_default_params({"ncpus": 1})

/cluster2/huanglab/jiamao/conda/envs/celloracle/lib/python3.10/site-packages/gimmemotifs/config.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm

In [3]:
import celloracle as co
from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
co.__version__

which: no R in (/cluster2/huanglab/jiamao/conda/envs/celloracle/bin:/cluster/home/jiamao/.cursor-server/bin/b3573281c4775bfc6bba466bf6563d3d498d1070/bin/remote-cli:/opt/simplehpc/scm/bin:/opt/simplehpc/scm/bin:/cluster/home/jiamao/.cargo/bin:/opt/simplehpc/scm/bin:/opt/simplehpc/scm/bin:/cluster2/huanglab/jiamao/conda/envs/celloracle/bin:/cluster2/huanglab/jiamao/conda/condabin:/opt/simplehpc/scm/bin:/usr/share/Modules/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin:/cluster/home/jiamao/.local/bin:/cluster/home/jiamao/bin:/cluster2/huanglab/jiamao/Apps/HOMER/bin:/cluster2/huanglab/jiamao/Apps/liftOver:/cluster/home/jiamao/.local/bin:/cluster/home/jiamao/bin:/cluster2/huanglab/jiamao/Apps/HOMER/bin:/cluster2/huanglab/jiamao/Apps/liftOver)


'0.20.0'

In [4]:
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = (15,7)
plt.rcParams["savefig.dpi"] = 600

# Gernerate GRN from Scenicplus

In [ ]:
# tf_to_gene_adj.tsv from Scenicplus
tf_to_gene_adj = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Oligo/outs/tf_to_gene_adj.tsv", sep="\t")
tf_to_gene_adj.shape

(2404491, 5)

In [ ]:
# region_to_gene_adj.tsv from Scenicplus
region_to_gene_adj = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Oligo/outs/region_to_gene_adj.tsv", sep="\t")
region_to_gene_adj.shape

(422371, 7)

In [41]:
import pandas as pd
import ast 

region_df = region_to_gene_adj.copy()
# prepare the distance column
def parse_distance(x):
    if isinstance(x, str):
        try:
            return abs(ast.literal_eval(x)[0])
        except:
            return 999999999
    elif isinstance(x, list):
        return abs(x[0])
    return abs(x)

region_df['abs_distance'] = region_df['Distance'].apply(parse_distance)

# set the filtering threshold
# condition 1: must be positively correlated (Open chromatin -> Gene Expression)
# set a slightly higher threshold 0.03, filter out very weak correlations
cond_rho = region_df['rho'] > 0.03 

# condition 2: Importance must be high
# strategy: take the 50th percentile of Importance (keep top 50% strong connections)
importance_cutoff = region_df['importance'].quantile(0.5)
cond_imp = region_df['importance'] > importance_cutoff

# condition 3: distance should not be too far
cond_dist = region_df['abs_distance'] < 500000


# filtering
region_filtered = region_df[cond_rho & cond_imp & cond_dist].copy()
print(f"original number of connections: {len(region_df)}")
print(f"number of connections after filtering: {len(region_filtered)}")

# preview
region_filtered.head()

original number of connections: 422371
number of connections after filtering: 61474


,target,region,importance,rho,importance_x_rho,importance_x_abs_rho,Distance,abs_distance
14,A2ML1,chr12:8689692-8690193,0.084664,0.069975,0.005924,0.005924,[132679],132679
16,A2ML1,chr12:8687707-8688208,0.104774,0.066157,0.006932,0.006932,[134663],134663
17,A2ML1,chr12:8697641-8698142,0.052208,0.036050,0.001882,0.001882,[124729],124729
21,A2ML1,chr12:8914190-8914691,0.044473,0.036853,0.001639,0.001639,[-27439],27439
23,A2ML1,chr12:8698402-8698903,0.059290,0.042637,0.002528,0.002528,[123969],123969


In [44]:
tf_filtered = tf_to_gene_adj[tf_to_gene_adj['regulation'].isin([1, -1])].copy()
region_df = region_filtered.copy()

# Merge the region and TF information
# this will produce a long table: Region - Gene - TF
merged_df = pd.merge(
    region_df[['region', 'target']],  # left table: Region and Gene
    tf_filtered[['TF', 'target']],    # right table: TF and Gene
    on='target',                      # join key
    how='inner'                       # take the intersection (i.e., the gene must have both Region and TF)
)

# fill the value with 1 (connection)
merged_df['value'] = 1 

# fill the missing value with 0
oligo_GRN = merged_df.pivot_table(
    index=['region', 'target'], 
    columns='TF', 
    values='value', 
    fill_value=0
)

# format adjustment (adapt to CellOracle format)
oligo_GRN = oligo_GRN.reset_index()


# gene_short_name corresponds to target
oligo_GRN = oligo_GRN.rename(columns={
    'region': 'peak_id',
    'target': 'gene_short_name'
})

# check the result
print(f"generated matrix dimension: {oligo_GRN.shape}")

# ensure the data type is float (CellOracle sometimes requires float)
oligo_GRN.iloc[:, 2:] = oligo_GRN.iloc[:, 2:].astype(float)
oligo_GRN.head()

generated matrix dimension: (61474, 705)


TF,peak_id,gene_short_name,ACAA1,ADNP2,AHCTF1,ALX1,ALX3,ALX4,AR,ARID3A,...,ZNF860,ZNF875,ZNF880,ZNF90,ZNF91,ZNF93,ZNF98,ZSCAN22,ZSCAN26,ZSCAN29
0,chr10:100009054-100009555,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10:100020821-100021322,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr10:100025575-100026076,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10:100073098-100073599,CHUK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,chr10:100246930-100247431,CHUK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [48]:
oligo_GRN.columns.name = None

In [49]:
oligo_GRN.to_parquet("oligo_GRN_dataframe.parquet")

## 2.1. Check data format

Here, the function below will check peak data format, including chromosome name and peak length.

In [7]:
peaks = ma.check_peak_format(peaks, ref_genome, genomes_dir='/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/CellOracle/tutorials/reference')

Peaks before filtering:  15779
Peaks with invalid chr_name:  0
Peaks with invalid length:  2
Peaks after filtering:  15777


## 2.2. [Optional step] Load custom motifs

You can chose to use either a custom TF binding reference or CellOracle’s default motifs during the motif analysis. If you would like to use our default motifs, you can continue to the next step without loading any additional data.


If you would like to use a custom motif dataset, please choose one of the following options.

- Motifs provided by gimmemotifs
 >Gimmemotifs is a python package for motif analysis. It provides many motif dataset. https://gimmemotifs.readthedocs.io/en/master/overview.html#motif-databases
 > 
 > Please use this notebook to learn how to load motif data from gimmemotifs database. 
 > https://github.com/morris-lab/CellOracle/blob/master/docs/notebooks/02_motif_scan/motif_data_preparation/01_How_to_load_gimmemotifs_motif_data.ipynb

- Custom motifs provided by CellOracle.
 
 >CellOracle also provides many motif datasets generated from CisBP. http://cisbp.ccbr.utoronto.ca/
 >
 >Please look at this notebook to learn how to load the CisBP motifs.https://github.com/morris-lab/CellOracle/blob/master/docs/notebooks/02_motif_scan/motif_data_preparation/02_How_to_load_CisBPv2_motif_data.ipynb


- Make your own custom motif data.
 >You can create custom motif data by yourself.
 >
 >Please look at this notebook to learn how to create your custom motif dataset.https://github.com/morris-lab/CellOracle/blob/master/docs/notebooks/02_motif_scan/motif_data_preparation/03_How_to_make_custom_motif.ipynb


# 3. Instantiate TFinfo object and search for TF binding motifs
The motif analysis module has a custom class, `TFinfo`. 
The TFinfo objectexecutes the steps below.

- Converts a peak data into a DNA sequences.
- Scans the DNA sequences searching for TF binding motifs.
- Post-processes the motif scan results.
- Converts data into appropriate format. You can convert data into base-GRN. The base GRN data can be formatted as either a python dictionary or pandas dataframe. This output will be the final base GRN used in the GRN model construction step.

## 3.1. Instantiate TFinfo object
If your reference genome file are installed in non-default location, please speficy the location using `genomes_dir`.

In [13]:
peaks[1:20]

,peak_id,gene_short_name
1,chr10_100486534_100488209,Tmtc3
2,chr10_100588641_100589556,4930430F08Rik
3,chr10_100741247_100742505,Gm35722
4,chr10_101681379_101682124,Mgat4c
5,chr10_102158688_102159257,Mgat4c
6,chr10_102511934_102512015,Rassf9
7,chr10_103026814_103029423,Alx1
8,chr10_103235705_103236587,Lrriq1
9,chr10_103366977_103369690,Slc6a15
10,chr10_10472105_10472772,Adgb


In [8]:
# Instantiate TFinfo object
tfi = ma.TFinfo(peak_data_frame=peaks[1:20], 
                ref_genome=ref_genome,
                genomes_dir='/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/CellOracle/tutorials/reference') 

## 3.2. Motif scan


You can specify the TF binding motif data as follows. 

`tfi.scan(motifs=motifs)`

If you do not specify the motifs or set motifs to `None`, the default motifs will be loaded automatically.

- For mouse and human, "gimme.vertebrate.v5.0." will be used as the default motifs. 

- For another species, the species-specific TF binding motif data extracted from CisBP ver2.0 will be used.



**If your jupyter notebook kernel is killed during the motif scan process, please see the link below.**

https://morris-lab.github.io/CellOracle.documentation/installation/python_step_by_step_installation.html#install-gimmemotifs-with-conda

In [17]:
from norns import config as cfg
config = cfg("genomepy", default="config/default.yaml")
config.update({"genomes_dir": "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/CellOracle/tutorials/reference"})

In [9]:
%%time
# Scan motifs. !!CAUTION!! This step may take several hours if you have many peaks!
tfi.scan(fpr=0.02, 
         motifs=None,  # If you enter None, default motifs will be loaded.
         verbose=True)

# Save tfinfo object
tfi.to_hdf5(file_path="test1.celloracle.tfinfo")

No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-12-18 09:21:17,456 - DEBUG - using background: genome /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/CellOracle/tutorials/reference/mm10 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 



2025-12-18 09:21:31,060 - DEBUG - determining FPR-based threshold


Motif scan started .. It may take long time.



Scanning:   0%|          | 0/17 [00:00<?, ? sequences/s]

CPU times: user 3min 11s, sys: 7.71 s, total: 3min 19s
Wall time: 3min 36s


In [10]:
# Check motif scan results
tfi.scanned_df.head()

,seqname,motif_id,factors_direct,factors_indirect,score,pos,strand
0,chr10_100486534_100488209,GM.5.0.Mixed.0001,,"SRF, EGR1",8.491542,1037,1
1,chr10_100486534_100488209,GM.5.0.Mixed.0001,,"SRF, EGR1",7.271743,1065,-1
2,chr10_100486534_100488209,GM.5.0.Nuclear_receptor.0002,NR2C2,"NR2C2, Nr2c2",9.258236,1160,-1
3,chr10_100486534_100488209,GM.5.0.Nuclear_receptor.0002,NR2C2,"NR2C2, Nr2c2",9.190379,783,1
4,chr10_100486534_100488209,GM.5.0.C2H2_ZF.0001,"Egr1, EGR, EGR1, Egr2, EGR2","Egr4, EGR1",8.925386,387,1


We have the score for each sequence and motif_id pair.
In the next step we will filter the motifs with low scores.

# 4. Filtering motifs

In [11]:
# Reset filtering 
tfi.reset_filtering()

# Do filtering
tfi.filter_motifs_by_score(threshold=10)

# Format post-filtering results.
tfi.make_TFinfo_dataframe_and_dictionary(verbose=True)



Filtering finished: 9435 -> 2246
1. Converting scanned results into one-hot encoded dataframe.


  0%|          | 0/17 [00:00<?, ?it/s]

2. Converting results into dictionaries.


  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/811 [00:00<?, ?it/s]

# 5. Get final base GRN

## 5.1. Get results as a dataframe

In [12]:
df = tfi.to_dataframe()
df.head()

,peak_id,gene_short_name,9430076c15rik,Ac002126.6,Ac012531.1,Ahr,Ahrr,Aire,Al592170.2,Al662828.6,...,Znf75a,Znf75d,Znf76,Znf768,Znf770,Znf784,Znf8,Zscan22,Zscan31,Zscan4
0,chr10_100486534_100488209,Tmtc3,0,0,0,0,0,0,0,0,...,0,0,1,0,0,1,0,1,0,0
1,chr10_100588641_100589556,4930430F08Rik,0,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,chr10_100741247_100742505,Gm35722,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,chr10_101681379_101682124,Mgat4c,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,chr10_102158688_102159257,Mgat4c,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 6. Save results
We will use this information when constructing the GRN models later. Save the results.

In [13]:
# Save result as a dataframe
df = tfi.to_dataframe()
df.to_parquet("base_GRN_dataframe.parquet")


In [14]:
df

,peak_id,gene_short_name,9430076c15rik,Ac002126.6,Ac012531.1,Ahr,Ahrr,Aire,Al592170.2,Al662828.6,...,Znf75a,Znf75d,Znf76,Znf768,Znf770,Znf784,Znf8,Zscan22,Zscan31,Zscan4
0,chr10_100486534_100488209,Tmtc3,0,0,0,0,0,0,0,0,...,0,0,1,0,0,1,0,1,0,0
1,chr10_100588641_100589556,4930430F08Rik,0,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,chr10_100741247_100742505,Gm35722,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,chr10_101681379_101682124,Mgat4c,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,chr10_102158688_102159257,Mgat4c,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,chr10_102511934_102512015,Rassf9,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,chr10_103026814_103029423,Alx1,1,1,0,0,0,1,1,0,...,0,0,0,0,1,0,0,1,0,0
7,chr10_103235705_103236587,Lrriq1,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
8,chr10_103366977_103369690,Slc6a15,0,0,0,1,1,0,0,1,...,0,0,0,1,0,0,0,1,0,0
9,chr10_10472105_10472772,Adgb,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**We will use this base GRN data in the GRN construction section.**

https://morris-lab.github.io/CellOracle.documentation/tutorials/networkanalysis.html